# 직접 수집 3 · 데이터 수집 ② — XTools 지표

필터링된 각 문서의 **활동 지표**를 수집합니다. 지표 원천은 **XTools**(위키미디어 통계 도구) + 위키 API:
- **편집수(edit)** · **조회수(pageviews)** · **링크(link)** · **문서 정보(info: 생성일·분류 등)**

이 값들이 나중에 **기술집약도·수요/공급 부상성** 지표로 환산됩니다.
> ⚠️ 문서마다 4종 호출이 있어 **가장 오래 걸리는 단계**입니다(수 분). 이미 수집된 경우 캐시로 건너뜁니다.

In [2]:
# --- 공통 준비: 프로젝트 루트로 이동 + 임포트 + 경로 정의 ---
import os, sys, warnings
warnings.filterwarnings("ignore")
for _ in range(5):                       # wiki_crawling.py 있는 프로젝트 루트 탐색
    if os.path.isfile("wiki_crawling.py"):
        break
    os.chdir("..")
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import pandas as pd
import pipeline as P
import wiki_crawling as wc

SEED = "Quantum computing"               # ← 분석할 위키 문서 (자유 변경)
N_DEPTH = 2                              # 1차수 고정
slug = P.slugify_seed(SEED)
BASE = os.path.join("runs", slug)
os.makedirs(os.path.join(BASE, "seed_item"), exist_ok=True)
os.makedirs(os.path.join(BASE, "xtools_item", slug), exist_ok=True)
EXPAND   = os.path.join(BASE, "seed_item", f"{N_DEPTH}차시 확장 최종_결과.xlsx")
FILTER   = os.path.join(BASE, "seed_item", f"{slug}_filtering_network.xlsx")
XTOOLS   = os.path.join(BASE, "xtools_item", slug)
FLAG     = os.path.join(XTOOLS, ".collect_done")
PAGERANK = os.path.join(BASE, f"{slug}_pagerank.xlsx")
STATS    = os.path.join(BASE, f"{slug}_statistics.xlsx")
print("작업 루트:", os.getcwd())
print("대상 문서:", SEED, "| 작업 폴더:", BASE)

작업 루트: /Users/sumin/Desktop/유망 아이템/wiki_web
대상 문서: Quantum computing | 작업 폴더: runs/Quantum_computing


In [3]:
# [이전 단계 재현] 시드 실제 제목 (캐시되어 즉시)
df_true = P.resolve_true_title(SEED, os.path.join(BASE, "true_title.xlsx"))
SEED_TRUE = P.title_space(str(df_true.loc[0, "True_title"]))
print("시드 실제 제목:", SEED_TRUE)

시드 실제 제목: Quantum computing


In [4]:
# [이전 단계 재현] 네트워크 확장 (캐시되어 즉시)
EXPAND = P.expand_network(SEED_TRUE, EXPAND, N_DEPTH)
print("확장 결과:", EXPAND)

[Quantum computing] 기존 확장 파일 발견 (alt): runs/Quantum_computing/seed_item/Quantum computing/2차시 확장 최종_결과.xlsx
확장 결과: runs/Quantum_computing/seed_item/Quantum computing/2차시 확장 최종_결과.xlsx


In [5]:
# [이전 단계 재현] 네트워크 필터링 (캐시되어 즉시)
FILTER = P.filter_network(EXPAND, SEED_TRUE, FILTER, mode="balanced")
print("필터 결과:", FILTER)

필터 결과: runs/Quantum_computing/seed_item/Quantum_computing_filtering_network.xlsx


## 1) 개별 XTools 호출 맛보기 — 시드의 연도별 편집수
전체 수집 전에, 한 문서의 편집 이력을 직접 불러 API 형태를 봅니다.

In [6]:
edit_df = wc.crawl_edit_by_year(SEED_TRUE)
print(f"'{SEED_TRUE}' 연도별 편집 데이터")
edit_df.head(12)

'Quantum computing' 연도별 편집 데이터


,title,edits,year
0,Quantum_computing,11,2001
1,Quantum_computing,23,2002
2,Quantum_computing,32,2003
3,Quantum_computing,194,2004
4,Quantum_computing,152,2005
5,Quantum_computing,237,2006
6,Quantum_computing,355,2007
7,Quantum_computing,285,2008
8,Quantum_computing,198,2009
9,Quantum_computing,118,2010


## 2) 전체 노드 수집 + 통합
`xtools_collect` 가 필터된 모든 노드의 4종 지표를 수집하고(문서별 파일),
`xtools_integrate` 가 이를 지표별 통합본으로 합칩니다.

In [7]:
def on_node(**kw):
    print(f"  수집: {kw.get('node')}")

P.xtools_collect(SEED_TRUE, FILTER, XTOOLS, FLAG, on_node=on_node)
outs = P.xtools_integrate(SEED_TRUE, XTOOLS)
print("\n통합 파일:", {k: os.path.basename(v) for k, v in outs.items()})

[Quantum computing] 편집/정보/링크/조회수 수집 중 (76개 노드)...
runs/Quantum_computing/xtools_item/Quantum_computing


 단어 수집 중....: Quantum weirdness (1/76)


  수집: Quantum weirdness


 단어 수집 중....: Photonic molecule (2/76)


  수집: Photonic molecule


...

  수집: Distributed computing


 단어 수집 중....: Analog computer (76/76)


  수집: Analog computer
[Quantum computing] 누락 노드 8개 재시도...
runs/Quantum_computing/xtools_item/Quantum_computing


 단어 수집 중....: Supercomputing in Taiwan (1/8)
 단어 수집 중....: Testing high-performance computing applications (2/8)
 단어 수집 중....: QBism (3/8)
 단어 수집 중....: Quantum tunnelling (4/8)
 단어 수집 중....: High-performance computing (5/8)
 단어 수집 중....: Supercomputing in China (6/8)
 단어 수집 중....: QpiAI-Indus (7/8)
 단어 수집 중....: Network computing (8/8)



통합 파일: {'edit': 'Quantum computing_all_edit.xlsx', 'info': 'Quantum computing_all_info.xlsx', 'link': 'Quantum computing_all_link.xlsx', 'pageviews': 'Quantum computing_all_pageviews.xlsx'}


## 3) 통합 결과 살펴보기 (편집 통합본 예시)

In [8]:
edit_all = pd.read_excel(outs["edit"]) if "edit" in outs else pd.DataFrame()
print("편집 통합본 shape:", edit_all.shape)
edit_all.head(10)

편집 통합본 shape: (76, 26)


,title,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,ACM SIGHPC,0,0,0,0,0,0,0,0,0,...,0,24,12,7,2,0,4,1,1,2
1,AQUA@home,0,0,0,0,0,0,0,0,10,...,4,1,2,0,2,1,11,0,1,7
2,Adiabatic quantum computation,0,0,0,0,0,0,8,6,11,...,6,5,4,3,8,7,3,3,1,6
3,Analog computer,3,9,24,48,61,108,132,72,101,...,37,58,34,59,39,25,33,65,46,27
4,Complex system,0,5,13,26,88,104,131,41,34,...,48,32,52,68,53,75,41,48,44,50
5,Computer,3,36,65,372,1099,2491,984,151,196,...,110,74,72,136,82,80,109,149,79,47
6,D-Wave Systems,0,0,0,0,0,0,128,20,22,...,48,73,72,80,27,15,16,21,16,47
7,Distributed computing,1,16,34,99,218,194,182,101,164,...,62,56,46,43,18,27,49,30,34,25
8,Electronic quantum holography,0,0,0,0,0,0,0,0,12,...,0,0,2,6,2,5,2,0,3,30
9,Fidelity of quantum states,0,0,0,0,0,13,4,4,7,...,3,4,18,2,5,7,2,17,5,10


**정리**
- 연관 문서 전체의 편집·조회·링크·정보를 수집·통합했습니다.
- 다음 노트북(**04 결과**)에서 이 원천 데이터를 지표로 환산하고 네트워크를 분석합니다.